# Assignment 3-4

### Виконали: Іван Лукʼянець, Андрій Бойчук

### Реалізація Геш-функцiї SHA-256

In [16]:
def rotr32(n, c):
    # Performs a cyclic right shift with removed bits appearing on the left
    return ((n >> c) | (n << (32 - c))) & 0xFFFFFFFF

def sha256(message):
    # K values given from the SHA-256 config file
    K = [
        0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5, 0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
        0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3, 0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
        0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc, 0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
        0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7, 0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
        0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13, 0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
        0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3, 0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
        0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5, 0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
        0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208, 0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2
    ]
    # Initial variable values
    h = [
        0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a,
        0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19
    ]

    # Padding
    msg = bytearray(message)
    orig_len_bits = len(msg) * 8
    msg.append(0x80)

    # Leave 64 bits for length-padding
    while (len(msg) * 8 + 64) % 512 != 0:
        msg.append(0x00)

    # Add length padding
    msg.extend(orig_len_bits.to_bytes(8, 'big'))

    # Blocks
    for i in range(0, len(msg), 64):
        chunk = msg[i:i+64]
        w = [0] * 64

        # First 16 elements of a block
        for t in range(16):
            w[t] = int.from_bytes(chunk[t*4:t*4+4], 'big')

        # Rest of 64 elements processed
        for t in range(16, 64):
            s0 = rotr32(w[t-15], 7) ^ rotr32(w[t-15], 18) ^ (w[t-15] >> 3)
            s1 = rotr32(w[t-2], 17) ^ rotr32(w[t-2], 19) ^ (w[t-2] >> 10)
            w[t] = (w[t-16] + s0 + w[t-7] + s1) & 0xFFFFFFFF

        # Init values
        a, b, c, d, e, f, g, current_h = h

        # 64 iterations of the hash-function
        for t in range(64):
            # Compute t1
            S1 = rotr32(e, 6) ^ rotr32(e, 11) ^ rotr32(e, 25)
            ch = (e & f) ^ (~e & g)
            t1 = (current_h + S1 + ch + K[t] + w[t]) & 0xFFFFFFFF

            # Compute t2
            S0 = rotr32(a, 2) ^ rotr32(a, 13) ^ rotr32(a, 22)
            maj = (a & b) ^ (a & c) ^ (b & c)
            t2 = (S0 + maj) & 0xFFFFFFFF

            # Shift values for next iteration + compute e and a
            current_h = g
            g = f
            f = e
            e = (d + t1) & 0xFFFFFFFF
            d = c
            c = b
            b = a
            a = (t1 + t2) & 0xFFFFFFFF
        # Get final values
        h[0] = (a + h[0]) & 0xFFFFFFFF
        h[1] = (b + h[1]) & 0xFFFFFFFF
        h[2] = (c + h[2]) & 0xFFFFFFFF
        h[3] = (d + h[3]) & 0xFFFFFFFF
        h[4] = (e + h[4]) & 0xFFFFFFFF
        h[5] = (f + h[5]) & 0xFFFFFFFF
        h[6] = (g + h[6]) & 0xFFFFFFFF
        h[7] = (current_h + h[7]) & 0xFFFFFFFF

    # Return hex elements in a string to ease the testing
    return "".join(f"{x:08x}" for x in h)

#### Перевірка тестових векторів

In [ ]:
assert sha256(b"abc") == "ba7816bf 8f01cfea 414140de 5dae2223 b00361a3 96177a9c b410ff61 f20015ad".replace(" ", ""), 'Test Case for "abc" failed'

assert sha256(b"") == "e3b0c442 98fc1c14 9afbf4c8 996fb924 27ae41e4 649b934c a495991b 7852b855".replace(" ", ""), 'Test Case for "" failed'

assert sha256(b"abcdbcdecdefdefgefghfghighijhijkijkljklmklmnlmnomnopnopq") == "248d6a61 d20638b8 e5c02693 0c3e6039 a33ce459 64ff2167 f6ecedd4 19db06c1".replace(" ", ""), 'Test Case for "abcdbcdecdefdefgefghfghighijhijkijkljklmklmnlmnomnopnopq" failed'

assert sha256(b"abcdefghbcdefghicdefghijdefghijkefghijklfghijklmghijklmnhijklmnoijklmnopjklmnopqklmnopqrlmnopqrsmnopqrstnopqrstu") == "cf5b16a7 78af8380 036ce59e 7b049237 0b249b11 e8f07a51 afac4503 7afee9d1".replace(" ", ""), 'Test Case for "abcdefghbcdefghicdefghijdefghijkefghijklfghijklmghijklmnhijklmnoijklmnopjklmnopqklmnopqrlmnopqrsmnopqrstnopqrstu" failed'

assert sha256(b"a" * 1000000) == "cdc76e5c 9914fb92 81a1c7e2 84d73e67 f1809a48 a497200e 046d39cc c7112cd0".replace(" ", ""), 'Test Case for "a" * 1000000 failed'

assert sha256(b"abcdefghbcdefghicdefghijdefghijkefghijklfghijklmghijklmnhijklmno" * 16777216) == "50e72a0e 26442fe2 552dc393 8ac58658 228c0cbf b1d2ca87 2ae43526 6fcd055e".replace(" ", ""), 'Test Case for "abcdefghbcdefghicdefghijdefghijkefghijklfghijklmghijklmnhijklmno" * 16,777,216 failed'


print("Tests were successfull!!!")


Цю ^^^ тєму не рань, останній тест на годину

### Підбір префікса

In [73]:
import numpy as np
import hashlib
import os

def find_prefix(message = b"give my friend 2 bitcoins for a pizza"):

  length = 20
  n_zeros = 32//8
  counter = 0

  while True:
    prefix = counter.to_bytes(20, 'big')
    # prefix = os.urandom(length)
    counter +=1
    message_pref = prefix + message
    result = hashlib.sha256(message_pref).digest()


    if result[:n_zeros] == b'\x00' * (n_zeros):
      return prefix, counter, result

In [74]:
prefix, counter, res_str = find_prefix()
print(f"Found the prefix {prefix} with {counter} steps. The result of the Hash function is {res_str}")

Found the prefix b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x9b\xdf\x14J' with 2615088203 steps. The result of the Hash function is b'\x00\x00\x00\x00\x16o\xa5\x05\xce\x1a\xc5\xe7\xf3\xef\\\xb2$X\x99\xdc \x0fC$m9\x89\x82GS\xca\xe4'


### Ключі та сертифікат

#### Створення власної пари секретного і відкритого ключа

In [1]:
%%capture
!openssl genpkey -algorithm RSA -out server.key -pkeyopt rsa_keygen_bits:8192

#### Запит на створення сертифікату з відкритим ключем та метаданими

In [2]:
!openssl req -new -key server.key -out server.csr -subj "/C=UA/L=Kyiv/O=KSE/CN=www.crypto.kse.ua emailAddress=aboichuk@kse.org.ua" -nodes

In [7]:
!openssl req -text -noout -in server.csr

Certificate Request:
    Data:
        Version: 1 (0x0)
        Subject: C=UA, L=Kyiv, O=KSE, CN=www.crypto.kse.ua emailAddress=aboichuk@kse.org.ua
        Subject Public Key Info:
            Public Key Algorithm: rsaEncryption
                Public-Key: (8192 bit)
                Modulus:
                    00:98:b2:c7:43:b2:df:53:5a:42:35:51:8a:ee:b1:
                    97:a2:ff:40:17:4e:97:b0:96:f5:63:95:61:0b:41:
                    77:b9:77:11:be:13:3a:15:da:f6:2d:57:04:c4:ab:
                    1c:a1:31:0c:7f:7c:54:c6:1b:98:89:d5:31:50:1a:
                    04:71:0b:7a:1a:10:1e:ea:9d:f9:b2:6f:6c:20:53:
                    26:fe:e8:3a:73:f7:b0:d3:d3:e1:33:be:1a:69:f5:
                    0d:55:c5:cd:9e:71:fd:49:15:86:6e:0b:3a:39:2a:
                    3e:35:ef:4e:3d:c3:e8:48:10:ea:e5:bf:5c:2c:77:
                    7b:99:f7:73:b8:bd:2e:62:71:94:7a:91:b2:f8:f8:
                    dd:0b:cf:a5:03:44:02:b1:27:89:3c:39:45:32:bb:
                    f4:58:80:26:c4:d2:3d:35:11:

#### Самопідписаний сертифікат

In [4]:
!openssl x509 -req -sha256 -in server.csr -signkey server.key -out server.crt

Certificate request self-signature ok
subject=C=UA, L=Kyiv, O=KSE, CN=www.crypto.kse.ua emailAddress=aboichuk@kse.org.ua


In [8]:
 !openssl x509 -text -noout -in server.crt

Certificate:
    Data:
        Version: 1 (0x0)
        Serial Number:
            2d:b5:fb:3c:eb:ad:1f:b0:c7:3e:e0:d6:64:f3:c2:b1:8c:6b:42:21
        Signature Algorithm: sha256WithRSAEncryption
        Issuer: C=UA, L=Kyiv, O=KSE, CN=www.crypto.kse.ua emailAddress=aboichuk@kse.org.ua
        Validity
            Not Before: May  2 14:07:50 2026 GMT
            Not After : Jun  1 14:07:50 2026 GMT
        Subject: C=UA, L=Kyiv, O=KSE, CN=www.crypto.kse.ua emailAddress=aboichuk@kse.org.ua
        Subject Public Key Info:
            Public Key Algorithm: rsaEncryption
                Public-Key: (8192 bit)
                Modulus:
                    00:98:b2:c7:43:b2:df:53:5a:42:35:51:8a:ee:b1:
                    97:a2:ff:40:17:4e:97:b0:96:f5:63:95:61:0b:41:
                    77:b9:77:11:be:13:3a:15:da:f6:2d:57:04:c4:ab:
                    1c:a1:31:0c:7f:7c:54:c6:1b:98:89:d5:31:50:1a:
                    04:71:0b:7a:1a:10:1e:ea:9d:f9:b2:6f:6c:20:53:
                    26:fe:e8:3a

#### Підписання сертифікату іншої команди

In [ ]:
!openssl x509 -req -in server_utkin_korcheniuk.csr -CA server.crt -CAkey server.key -CAcreateserial -out server_utkin_korcheniuk_signed.crt -sha256

In [11]:
!openssl x509 -text -noout -in server_utkin_korcheniuk_signed.crt

Certificate:
    Data:
        Version: 1 (0x0)
        Serial Number:
            11:48:9f:07:f7:f8:ec:9b:89:91:2a:46:71:c4:82:bd:81:32:1e:af
        Signature Algorithm: sha256WithRSAEncryption
        Issuer: C=UA, L=Kyiv, O=KSE, CN=www.crypto.kse.ua emailAddress=aboichuk@kse.org.ua
        Validity
            Not Before: May  2 14:11:10 2026 GMT
            Not After : Jun  1 14:11:10 2026 GMT
        Subject: C=UA, L=Kyiv, O=KSE, CN=www.crypto.kse.ua, emailAddress=user@kse.org.ua
        Subject Public Key Info:
            Public Key Algorithm: rsaEncryption
                Public-Key: (8192 bit)
                Modulus:
                    00:a1:ca:66:77:c0:f7:0b:ec:00:52:77:2f:86:da:
                    db:0a:bd:99:7d:dc:fd:5f:06:c6:b8:b2:7a:e8:a3:
                    61:0f:75:47:86:fb:41:db:e0:b3:fa:31:f1:49:6d:
                    5f:64:29:19:59:fb:ff:80:08:4f:4e:bb:e5:4a:2a:
                    ff:32:ec:9b:ed:39:a5:9e:91:5b:22:24:e8:1c:eb:
                    2c:ad:eb:be:84

### Пiдпиc повiдомлення «give my friend 2 bitcoins for a pizza»

In [13]:
!openssl dgst -sha256 -sign server.key -out pizza_signed.bin -sigopt rsa_padding_mode:pss -sigopt rsa_pss_saltlen:-1 pizza.txt

### Шифрування повiдомлення

Дістаємо відкритий ключ з самопідписаного сертифікату

In [14]:
!openssl x509 -pubkey -noout -in server.crt > key.pub

In [15]:
!openssl pkeyutl -encrypt -pubin -inkey key.pub -in pizza.txt -out pizza_encrypted.bin

### Відбиток чинного сертифікату КШЕ

Скачав сертифікат на цьому сайті https://crt.sh/?id=25341851341

In [22]:
with open("25341851341.crt", "rb") as f:
    crt_data = bytearray(f.read())

sha256(crt_data)

'81c45478e6683d60fa1af21c8f44e4b9fc15b18fa8dfb305452a29ad078912b6'

Як виявляється для знаходження хешу з сертифікату треба формат файлу der, тому перевів файл з формату crt до der

In [20]:
!openssl x509 -in 25341851341.crt -outform der -out final_cert.der

In [23]:
with open("final_cert.der", "rb") as f:
    der_data = f.read()

sha256(der_data)

'36a7601749fbad995a9779e2c8a2b6891d039870ff40e34be20da3a4bfccebe0'